In [1]:
import pandas as pd
from datasets import Dataset
!pip install -q -U transformers sentencepiece datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 43.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


In [2]:
file_path = "/kaggle/input/datasets/tanujsaxena/sandhi-data/sandhi_data.xlsx"

df = pd.read_excel(file_path)

df.head()
df = df.rename(columns={
    "Word": "input",
    "Split": "target"
})
df["target"] = df["target"].str.replace("+", " ", regex=False)
df["target"] = df["target"].str.strip()
df = df.dropna()
df = df[df["input"].str.strip() != ""]
df = df[df["target"].str.strip() != ""]
print(df.shape)

(13925, 2)


In [3]:
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2, seed=42)
test_val = dataset["test"].train_test_split(test_size=0.5, seed=42)

train_dataset = dataset["train"]
val_dataset = test_val["train"]
test_dataset = test_val["test"]

In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/mt5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [5]:
def preprocess(example):
    inputs = ["split sandhi: " + x for x in example["input"]]

    model_inputs = tokenizer(
        inputs,
        max_length=64,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        example["target"],
        max_length=64,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_dataset = train_dataset.map(preprocess, batched=True)
val_dataset = val_dataset.map(preprocess, batched=True)
test_dataset = test_dataset.map(preprocess, batched=True)

Map:   0%|          | 0/11140 [00:00<?, ? examples/s]

Map:   0%|          | 0/1392 [00:00<?, ? examples/s]

Map:   0%|          | 0/1393 [00:00<?, ? examples/s]

In [6]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./mt5-sandhi",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=10,
    learning_rate=3e-4,
    weight_decay=0.01,
    fp16=True,
    load_best_model_at_end=True,
    logging_steps=100,
    save_total_limit=2,
    optim="adamw_torch"
)

In [7]:
from transformers import Trainer, EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,1.706600,1.074170
2,0.530412,0.280464
3,0.305914,0.219514
4,0.253579,0.195901
5,0.212704,0.179770
6,0.198549,0.168978
7,0.184004,0.160820
8,0.171910,0.156415
9,0.161563,0.154000
10,0.164140,0.153709


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3490, training_loss=1.562063869705856, metrics={'train_runtime': 4516.1672, 'train_samples_per_second': 24.667, 'train_steps_per_second': 0.773, 'total_flos': 7362839445504000.0, 'train_loss': 1.562063869705856, 'epoch': 10.0})

In [8]:
import torch

def generate_predictions(dataset):
    predictions = []
    references = []

    model.eval()
    for example in dataset:
        input_text = "split sandhi: " + example["input"]

        input_ids = tokenizer(
            input_text,
            return_tensors="pt",
            truncation=True,
            max_length=40
        ).input_ids.to(model.device)

        with torch.no_grad():
            output = model.generate(input_ids, max_length=40)

        pred = tokenizer.decode(output[0], skip_special_tokens=True)

        predictions.append(pred.strip())
        references.append(example["target"].strip())

    return predictions, references

preds, refs = generate_predictions(test_dataset)


In [9]:
exact_matches = sum(p == r for p, r in zip(preds, refs))
accuracy = exact_matches / len(refs)

print("Exact Match Accuracy:", accuracy)


Exact Match Accuracy: 0.5922469490308686


In [10]:
from sklearn.metrics import f1_score

def word_level_f1(preds, refs):
    f1_scores = []
    for p, r in zip(preds, refs):
        p_tokens = p.split()
        r_tokens = r.split()

        common = set(p_tokens) & set(r_tokens)

        if len(common) == 0:
            f1_scores.append(0)
            continue

        precision = len(common) / len(p_tokens)
        recall = len(common) / len(r_tokens)

        f1 = 2 * precision * recall / (precision + recall)
        f1_scores.append(f1)

    return sum(f1_scores) / len(f1_scores)

print("Word-Level F1:", word_level_f1(preds, refs))


Word-Level F1: 0.7245534078269182


In [11]:
for i in range(10):
    print("Input :", test_dataset[i]["input"])
    print("Pred  :", preds[i])
    print("Gold  :", refs[i])
    print("-" * 40)


Input : नन्वित्यनुज्ञैषणायाम्
Pred  : ननुपूर्व अनुज्ञैषणायाम्
Gold  : ननु इति अनुज्ञैषणायाम्
----------------------------------------
Input : इत्‍थं च
Pred  : इत्थम् च
Gold  : इत्थम् च
----------------------------------------
Input : यो भाजनं भाजनं तं हत्वा
Pred  : यः भाजनं भाजनं तम् हत्वा
Gold  : यम् भाजनम् तम् हत्वा
----------------------------------------
Input : नान्त्यः
Pred  : न अान्त्यः
Gold  : न अन्त्यः
----------------------------------------
Input : ब्रह्मविद्यायां योगशास्त्रे
Pred  : ब्रह्मविद्यायाम् योगशास्त्रे
Gold  : ब्रह्मविद्यायाम् योगशास्त्रे
----------------------------------------
Input : किन्नाम
Pred  : किम् अनेन
Gold  : किं नाम
----------------------------------------
Input : शतं ह्यत्र
Pred  : शतम् ह्यत्र
Gold  : शतम् हि अत्र
----------------------------------------
Input : निर्माणमेव
Pred  : निर्माणम् एव
Gold  : निर्माणम् एव
----------------------------------------
Input : कथमित्याह
Pred  : कथम् इति आह
Gold  : कथम् इति आह
----------------------------------------
